# Smaller text-to-SQL prompts for open models with schema selection

_Authored by: [Ashish Sinha](https://github.com/ashishsinha1602)_

A text-to-SQL prompt carries the database schema, and every table in it costs context. That matters
more for open models than for hosted ones: a small model running on your own hardware has a short
context window, and every extra token slows generation down. On a real database most of the schema is
tables the question never touches.

In this notebook we send an open code model only the tables a question needs. The selection is done by
[schemagate](https://github.com/ashishsinha1602/schemagate), an Apache-2.0 library that ranks tables
offline (BM25 plus a small embedding model) and makes no model call itself. We will:

1. count the prompt tokens for the full schema and for the selection, with the model's own tokenizer;
2. generate SQL with [Qwen2.5-Coder-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-Coder-1.5B-Instruct)
   (Apache-2.0) from the selection, and run it against a real SQLite database;
3. see the selection respect who is asking: a table the caller may not read never enters the prompt;
4. look at when selection is worth it and what it can miss.

Everything runs on a CPU.

In [1]:
!pip install -q "schemagate>=1.14" transformers torch accelerate

## A schema with more tables than any question needs

schemagate ships a demo database: 42 tables and views across commerce, billing, HR and inventory, with
the traps real schemas have (near-duplicate tables, cryptic legacy names, join tables a question never
mentions). It is created as a SQLite file with a few rows, so generated SQL can actually run.

In [2]:
from schemagate import Principal
from schemagate.demo_schema import demo_catalog

catalog = demo_catalog()
objects = catalog.objects()
full_schema = "\n\n".join(o.render_ddl(max_columns=10**6) for o in objects)
print(len(objects), "tables and views,", len(full_schema), "characters of DDL")

42 tables and views, 8823 characters of DDL


## Full schema against selection, in the model's tokens

`select()` ranks the tables against the question, keeps the top six and adds the tables needed to join
them. A `Principal` says who is asking; the `okta:` prefix names where that identity came from.

In [3]:
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

QUESTION = "How many sales orders are there for each order status name?"
analyst = Principal("okta:analyst", roles={"finance"})


def messages(schema: str, question: str) -> list[dict]:
    return [
        {"role": "system", "content": "You write SQLite queries. Reply with one SELECT statement and nothing else."},
        {"role": "user", "content": f"Database schema:\n\n{schema}\n\nQuestion: {question}"},
    ]


def prompt_tokens(schema: str, question: str) -> int:
    ids = tokenizer.apply_chat_template(messages(schema, question), add_generation_prompt=True, return_dict=True)
    return len(ids["input_ids"])


selection = catalog.select(QUESTION, top_k=6, principal=analyst)
selected_schema = selection.prompt_fragment()

full = prompt_tokens(full_schema, QUESTION)
small = prompt_tokens(selected_schema, QUESTION)
print("tables sent:", [name.split(".")[-1] for name in selection.table_names])
print(f"full schema: {full:,} prompt tokens")
print(f"selection:   {small:,} prompt tokens ({1 - small / full:.0%} fewer)")

tables sent: ['sales_order_line', 'sales_order_status', 'v_order_fulfilment', 'sales_order', 'sup_purchase_order', 'sup_purchase_order_line', 'cat_product', 'sales_quote', 'core_address', 'crm_customer', 'sup_supplier']
full schema: 2,243 prompt tokens
selection:   725 prompt tokens (68% fewer)


## Generate SQL from the selection and run it

The SQL runs against the demo database, so we can check the answer is real and not just plausible. Small
models sometimes wrap the query in an explanation, so we take the SQL from the code block when there is
one, and if the database rejects the query we give the model the error and ask once for a fix.

In [4]:
import re
import sqlite3

import torch
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32)


def generate_sql(chat: list[dict]) -> str:
    inputs = tokenizer.apply_chat_template(chat, add_generation_prompt=True, return_tensors="pt", return_dict=True)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=200, do_sample=False)
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    fenced = re.search(r"```(?:sql)?\s*(.*?)```", text, re.S)
    return (fenced.group(1) if fenced else text).strip().rstrip(";")


db = sqlite3.connect(catalog._demo_url.removeprefix("sqlite:///"))
chat = messages(selected_schema, QUESTION)
sql = generate_sql(chat)
try:
    rows = db.execute(sql).fetchall()
except sqlite3.Error as err:
    print("first attempt failed:", err)
    chat += [{"role": "assistant", "content": sql},
             {"role": "user", "content": f"That query failed with: {err}. Write a corrected query."}]
    sql = generate_sql(chat)
    rows = db.execute(sql).fetchall()

print(sql)
print(len(rows), "rows, first five:")
for row in rows[:5]:
    print(tuple(round(v, 2) if isinstance(v, float) else v for v in row))

SELECT COUNT(*) AS count, T2.label FROM main.sales_order AS T1 INNER JOIN main.sales_order_status AS T2 ON T1.id_status = T2.id GROUP BY T2.label
4 rows, first five:
(28, 'Cancelled')
(112, 'Closed')
(29, 'Confirmed')
(71, 'Shipped')


## Selection is also access control

In the demo, `hr_compensation` is readable only by the `payroll` role. For anyone else it is not ranked
low; it is absent from the candidates, so the model cannot write a query against it or learn that it
exists.

In [5]:
SALARY = "salary and pay grade by employee"
for who in (analyst, Principal("okta:hr", roles={"payroll"})):
    names = [name.split(".")[-1] for name in catalog.select(SALARY, top_k=6, principal=who).table_names]
    print(sorted(who.roles), "sees hr_compensation:", "hr_compensation" in names)

['finance'] sees hr_compensation: False
['payroll'] sees hr_compensation: True


## When it is worth it, and what it can miss

The schemagate project measured the same comparison (full schema against selection, no model call) on
public benchmark databases. The median saving per database:

| Schemas | Median tables | Median saving |
|---|---|---|
| BIRD dev | 7 | 8.8% |
| Spider 2.0-lite BigQuery and Snowflake | 11 | 44.3% |
| Spider 2.0-lite databases with 50 or more tables | | 91.1% |

A seven-table database is already small, so there is little to cut; selection pays off on large
schemas, which is also where a small model's context runs out first. Selection does not make a small
model smarter: in our tests the 1.5B model handled questions needing one join, like the one above, but
not "which customers owe us money", where it has to find the right balance view and join it to customers; a 7B model or larger is the
better choice there, and the selection keeps its prompt just as short. It has a cost: a table the
question needs can be left out. On the Spider 2.0 warehouses, every table the reference SQL uses was
selected for 86.7% of questions. Check recall on your own questions, and raise `top_k` if tables go
missing. The script and per-database numbers are in the project's
[BENCHMARKS.md](https://github.com/ashishsinha1602/schemagate/blob/main/BENCHMARKS.md#prompt-tokens-on-public-schemas-the-saving-grows-with-the-schema).

To point this at your own database, replace `demo_catalog()` with
`Catalog().bootstrap("postgresql://...")` (or a SQLite or Oracle URL) and restrict
tables with `catalog.restrict("table_name", ["role"])`.